## **Feature Engineering**

In [24]:
from pathlib import Path
import pandas as pd
import numpy as np

## Load the Data from the data creation script

In [25]:
data = Path.cwd().parent / Path("data/simulated/simulated_transactions_seed_42.csv")

df = pd.read_csv(data)
df.head()

,tx_id,timestamp,user_id,amount,category,device_id,auth_method,lat,lon,ip_address,is_fraud
0,e4fd6c34-d67,2026-01-19 09:59:35.248955,user_55,172.43,grocery,475287aa,PIN,-83.615968,174.348649,120.185.31.102,0
1,0cb5ee5c-62f,2026-01-18 11:55:15.356815,user_407,8.43,food,379d58f9,Biometric,20.500359,150.789724,107.109.217.82,0
2,39a0f145-a5b,2026-02-14 00:45:15.799367,user_142,1259.07,tech,bea29dfe,PIN,-87.864685,123.877867,145.56.170.82,0
3,052226b9-836,2026-01-16 11:40:03.965444,user_492,26.14,grocery,052dfb9a,Biometric,-0.992383,-52.469217,208.72.228.108,0
4,a5ba2472-21d,2026-01-26 15:56:09.026376,user_261,364.83,utilities,0d18ab95,Biometric,8.142126,36.488989,104.34.170.221,0


## Feature engineering

In [26]:
# Convert 'timestamp' to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Ensure correct ordering - critical for time-based features
df = df.sort_values(["user_id", "timestamp"]).reset_index(drop=True)

In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   tx_id        10000 non-null  str           
 1   timestamp    10000 non-null  datetime64[us]
 2   user_id      10000 non-null  str           
 3   amount       10000 non-null  float64       
 4   category     10000 non-null  str           
 5   device_id    10000 non-null  str           
 6   auth_method  10000 non-null  str           
 7   lat          10000 non-null  float64       
 8   lon          10000 non-null  float64       
 9   ip_address   10000 non-null  str           
 10  is_fraud     10000 non-null  int64         
dtypes: datetime64[us](1), float64(3), int64(1), str(6)
memory usage: 1.4 MB


### Define functions to generate new features

In [ ]:
# ================================================================================================
# Haversine distance (km)
# ================================================================================================
def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate great-circle distance between two points on Earth.
    Inputs are in degrees. Output is in kilometers.
    """
    R = 6371.0  # Earth radius in km

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


# ================================================================================================
# Feature engineering
# ================================================================================================
def engineer_features(df):
    # ------------------------------------------------------------------------------------------
    # 0. Copy dataframe
    # ------------------------------------------------------------------------------------------

    # Ensure we don't modify the original dataframe(raw data)
    df = df.copy()

    # ------------------------------------------------------------------------------------------
    # 1. Time-based features
    # ------------------------------------------------------------------------------------------

    # Extract hour of day and day of week

    # Because fraud patterns may vary by time windows
    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek

    # ------------------------------------------------------------------------------------------
    # 2. Transaction velocity
    # ------------------------------------------------------------------------------------------

    
    counts = (
        df.groupby("user_id")
        .rolling("24h", on="timestamp", closed="left")["tx_id"]
        .count()
        .reset_index(level=0, drop=True) # Drops the index on 'user_id' level
        .values
    )

    # Assign back
    df["tx_count_24h"] = counts
    df["tx_count_24h"] = df["tx_count_24h"].fillna(0)
    
    # ------------------------------------------------------------------------------------------
    # 3. Behavioral features
    # ------------------------------------------------------------------------------------------

    # Calculate average spend per user up to (but not including) current transaction
    df["avg_spend_user"] = df.groupby("user_id")["amount"].transform(
        lambda x: x.shift(1).expanding().mean()
    )

    # amount to average spenditure ratio - handle division by zero

    # Because a ratio significantly higher than 1 (e.g., spending 10x the usual amount)
    # is a strong fraud signal.
    df["amount_ratio"] = np.where(
        df["avg_spend_user"] > 0, df["amount"] / df["avg_spend_user"], 0
    )

    # ------------------------------------------------------------------------------------------
    # 4. Geospatial features
    # ------------------------------------------------------------------------------------------

    # Previous transaction coordinates and timestamp
    df["prev_lat"] = df.groupby("user_id")["lat"].shift(1)
    df["prev_lon"] = df.groupby("user_id")["lon"].shift(1)
    df["prev_ts"] = df.groupby("user_id")["timestamp"].shift(1)

    # Distance from previous transaction (km)
    df["dist_from_last_tx_km"] = haversine(
        df["lat"], df["lon"], df["prev_lat"], df["prev_lon"]
    ).fillna(0)

    # Calculate time difference in hours (ensure it's not zero to avoid Inf)
    time_diff = (df["timestamp"] - df["prev_ts"]).dt.total_seconds() / 3600.0

    # If time passed is > 0, calculate speed. Otherwise, velocity is 0.
    df["travel_velocity_kmph"] = np.where(
        time_diff > 0, df["dist_from_last_tx_km"] / time_diff, 0
    )
    # --------------------------------------------------------------------------------------------
    # 5. Fraud rate features
    # --------------------------------------------------------------------------------------------

    df["fraud_rate_user"] = (
        df.groupby("user_id")["is_fraud"]
        .apply(lambda x: x.shift(1).expanding().mean())
        .reset_index(level=0, drop=True)
        .fillna(0)
    )

    df["fraud_rate_device"] = (
        df.groupby("device_id")["is_fraud"]
        .apply(lambda x: x.shift(1).expanding().mean())
        .reset_index(level=0, drop=True)
        .fillna(0)
    )

    # ------------------------------------------------------------------------------------------
    # 6. Categorical encoding
    # ------------------------------------------------------------------------------------------

    df = pd.get_dummies(df, columns=["auth_method", "category"], drop_first=True)

    # ------------------------------------------------------------------------------------------
    # 7. Drop non-model columns
    # ------------------------------------------------------------------------------------------

    df = df.drop(
        columns=[
            "tx_id",
            "prev_lat",
            "prev_lon",
            "prev_ts",
            "timestamp",
            "user_id",
            "device_id",
            "ip_address",
        ]
    )

    # ------------------------------------------------------------------------------------------
    # 8. Final numeric safety
    # ------------------------------------------------------------------------------------------
    num_cols = df.select_dtypes(include="number").columns
    df[num_cols] = df[num_cols].fillna(0)

    return df

## Call the function to generate the data

In [29]:
df = engineer_features(df)
df.tx_count_24h.nunique()

7

# **Save the feature into a csv**

In [30]:
ROOT_DIR = Path.cwd().parent
feature_dir = ROOT_DIR / "data/features"
feature_dir.mkdir(parents=True, exist_ok=True)

file_path = feature_dir / "fraud_features_seed_42.csv"
df.to_csv(file_path, index=False)
